## Cuaderno auxiliar 

In [4]:
import numpy as np

## Generando la Red Neuronal

### Estructuras de datos.

Para generalizar el modelo a un número arbitrario de capas, los **pesos** y **sesgos** se almacenan en una **lista de diccionarios**, donde cada elemento de la lista corresponde a una capa de la red. Esta representación es escalable y permite recorrer la red con bucles sin depender de nombres de variables específicos.

Si la arquitectura tiene dimensiones  
$$
(n_0, n_1, n_2, \dots, n_L),
$$
donde $n_0$ es el número de características de entrada y $n_L$ el número de neuronas de salida, entonces:

- Para cada capa $l = 1,\dots,L$:
  - Los pesos cumplen  
    $$
    W^{[l]} \in \mathbb{R}^{\,n_l \times n_{l-1}}
    $$
  - Los sesgos cumplen  
    $$
    b^{[l]} \in \mathbb{R}^{\,n_l \times 1}
    $$

En código, los parámetros de la red se representan así:

```python
params = [
    {"W": W1, "b": b1},   # capa 1
    {"W": W2, "b": b2},   # capa 2
    ...
    {"W": WL, "b": bL}    # capa L
]
```
### Ventajas de esta representación

- Permite recorrer todas las capas con un simple bucle `for`, sin necesidad de construir nombres como `"W1"`, `"W2"`, etc.
- Mantiene una correspondencia clara entre la notación matemática y el código:
  - `params[l]["W"]` ↔ $ W^{[l+1]} $
  - `params[l]["b"]` ↔ $ b^{[l+1]} $
- Facilita crear estructuras paralelas para el resto del entrenamiento:
  - `caches[l]` para almacenar los valores usados en el forward.
  - `grads[l]` para almacenar los gradientes en el backward.
- Escala naturalmente a arquitecturas profundas sin necesidad de modificar la estructura base.
- Reduce el riesgo de errores al trabajar con índices y nombres de parámetros, ya que cada capa queda contenida en un único diccionario.


### ¿Por qué crear una lista de *caches*?

Cada capa de la red se implementa para que **solo reciba los datos estrictamente necesarios** para realizar su parte del cálculo en el *forward* y su correspondiente derivada en el *backward*. Esto significa que una capa no debe depender de variables globales ni de valores internos de otras capas: únicamente necesita acceder a los elementos que intervienen directamente en sus fórmulas.

En las expresiones matemáticas que hemos visto en clase aparecen, para cada capa $l$:

- Los pesos $W^{[l]}$
- Los sesgos $b^{[l]}$
- Las activaciones de la capa anterior $A^{[l-1]}$
- La preactivación $Z^{[l]}$

Estas cantidades son necesarias en el *backward* para calcular:

$$
dW^{[l]},\quad db^{[l]},\quad dA^{[l-1]},\quad \delta^{[l]} = dZ^{[l]}
$$

Sin embargo, durante el *forward* las capas solo generan $Z^{[l]}$ y $A^{[l]}$, y después continúan hacia la siguiente capa. Si no almacenáramos esos valores en algún sitio, el *backward* no podría recuperarlos sin volver a ejecutar de nuevo toda la propagación hacia delante.

Por esta razón se crea una **lista de caches**, donde cada posición contiene un diccionario con la información necesaria para reconstruir las derivadas de esa capa:

```python
cache[l] = {
    "A_prev": A_prev,   # A^{[l-1]}
    "W": W,             # W^{[l]}
    "b": b,             # b^{[l]}
    "Z": Z              # Z^{[l]}
}
```

Cada capa añade su propio cache_l a una lista llamada caches, de manera que:

 - `caches[0]` corresponde a la capa 1

 - `caches[1]` corresponde a la capa 2
 
    …

 - `caches[L-1]` corresponde a la capa L

En el backward, se recorren estos caches en orden inverso, permitiendo que cada capa recupere exactamente las variables de sus fórmulas sin interferir con las demás.

Así, la estructura caches es indispensable para mantener el diseño modular de la red: cada capa calcula lo suyo con lo que le corresponde, y nada más.

### Funciones auxiliares

### Inicialización de los parámetros de la red neuronal

La función `initialize_params` se encarga de crear los **pesos** y **sesgos** de cada capa de la red neuronal a partir de la arquitectura definida en `layer_dims`.

Recordemos que:

- `layer_dims` es una lista con la dimensión de cada capa, por ejemplo:  
  $$
  [12288,\; 20,\; 7,\; 5,\; 1]
  $$
  donde $n_0 = 12288$ es la dimensión de entrada y $n_L = 1$ la dimensión de salida.

La función realiza los siguientes pasos:

1. **Opcionalmente fija la semilla** (`seed`) para que la inicialización sea reproducible.  
2. Recorre todas las capas (excepto la de entrada).  
3. Inicializa para cada capa $l$:

   - La matriz de pesos  
     $$
     W^{[l]} \in \mathbb{R}^{\,n_l \times n_{l-1}}
     $$
     usando la inicialización **He**:  
     $$
     W^{[l]} \sim \mathcal{N}(0,\; \sqrt{2/n_{l-1}})
     $$
     recomendada cuando se usa ReLU en capas ocultas.

   - El vector de sesgos  
     $$
     b^{[l]} = \vec{0}
     \in \mathbb{R}^{\,n_l \times 1}
     $$

4. Almacena cada par $(W, b)$ en una lista de diccionarios, uno por capa.

El resultado es una estructura de datos del tipo:

```python
[
  {"W": W1, "b": b1},
  {"W": W2, "b": b2},
  ...
  {"W": WL, "b": bL}
]
```
que luego será utilizada durante el forward y el backpropagation.

Esta inicialización es clave para que el entrenamiento sea estable y para evitar problemas como gradientes que explotan o desaparecen.


In [1]:
def initialize_params(layer_dims, seed=None):
    """
    layer_dims: lista [n_0, n_1, ..., n_L]
    devuelve: lista de diccionarios [{"W": W0, "b": b0}, ..., {"W": W_{L-1}, "b": b_{L-1}}]
    """
    if seed is not None:
        np.random.seed(seed)

    params = []
    for l in range(1, len(layer_dims)):
        n_prev = layer_dims[l-1]
        n_curr = layer_dims[l]
        layer = {
            "W": np.random.randn(n_curr, n_prev) * np.sqrt(2.0 / n_prev),
            "b": np.zeros((n_curr, 1))
        }
        params.append(layer)
    return params




#### Propagación hacia delante (*Feed Forward*)

La propagación hacia delante implementa exactamente las ecuaciones del PDF, que para cada capa $l$ son:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$
$$
A^{[l]} = g^{[l]}(Z^{[l]})
$$

Estas operaciones se repiten secuencialmente desde la capa 1 hasta la capa $L$. Para traducir estas ecuaciones al código, se implementan tres funciones clave: una función **lineal**, una función de **activación**, y una función que **combina ambas** para simplificar el bucle de forward.

---

1. `linear_forward(A_prev, W, b)`

Implementa la ecuación lineal:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$

- `A_prev` representa $A^{[l-1]}$
- `W` es $W^{[l]}$
- `b` es $b^{[l]}$

La función devuelve:

- `Z`: la preactivación de la capa  

---
2. Funciones de activación

Cada capa aplica una función escalar vectorizada:

$$
A^{[l]} = g^{[l]}(Z^{[l]})
$$

Las activaciones típicas son:

- **ReLU**:
  $$
  g(Z) = \max(0, Z)
  $$
- **Sigmoide**:
  $$
  g(Z) = \frac{1}{1 + e^{-Z}}
  $$

En código, cada función devuelve:
- `A`: activación $A^{[l]}$


---
3. `linear_activation_forward(A_prev, W, b, activation_fn)`

Esta función combina los dos pasos anteriores:

1. Cálculo lineal → $Z^{[l]} = W A^{[l-1]} + b$ 
2. Activación     → $A^{[l]} = g(Z^{[l]})$

La función devuelve:

- **`A`**: la activación de la capa, es decir $A^{[l]}$.
- **`cache`**: un diccionario que contiene todos los valores necesarios para la fase de *backpropagation*, en concreto:
  - `A_prev`: corresponde a $A^{[l-1]}$
  - `W`: corresponde a $W^{[l]}$
  - `b`: corresponde a $b^{[l]}$
  - `Z`: corresponde a $Z^{[l]}$

Este `cache` refleja exactamente las variables que aparecen en las ecuaciones para la derivación de los gradientes:

$$
dZ^{[l]}=\delta^{[l]},\quad 
dW^{[l]} = \delta^{[l]} A^{[l-1]\,T},\quad
db^{[l]} = \frac{1}{M}\sum \delta^{[l]},\quad
dA^{[l-1]} = W^{[l]\,T} \delta^{[l]}.
$$

Al guardar esta información durante el *forward*, cada capa dispone en el *backward* de todos los valores que necesita sin tener que recalcular nada ni acceder a datos de otras capas.






In [ ]:
def sigmoid(Z):   
    # ******** Tu codigo aqui ****************
    
    # ******** Tu codigo aqui ****************
    return A

def relu(Z):
    # ******** Tu codigo aqui ****************    
    
    # ******** Tu codigo aqui ****************
    return A

def linear_forward(A,W,b):
    # ******** Tu codigo aqui ****************
   
    # ******** Tu codigo aqui ****************
    return Z

def linear_activation_forward(A_prev, W, b, activation_fn):

    Z = linear_forward(A_prev, W, b)
    A = activation_fn(Z)

    cache = {"A_prev": A_prev,
                  "W": W,
                  "b": b,
                  "Z": Z}
    return A, cache
    

### Propagación hacia delante del modelo completo

La función `model_forward` implementa la **propagación hacia delante** (*forward propagation*) a través de **todas las capas** de la red neuronal.

Recordemos que en cada capa $l$ se realizan dos operaciones:

1. **Paso lineal**  
   $$
   Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
   $$
2. **Aplicación de la función de activación**  
   $$
   A^{[l]} = g^{[l]}(Z^{[l]})
   $$

donde $g^{[l]}$ puede ser ReLU, sigmoide, etc.



1. Inicializa `A` con la matriz de entrada `X`, que corresponde a $A^{[0]}$.
2. Recorre todas las capas de la red usando:
   - `params`: lista de diccionarios que contienen `W` y `b` de cada capa
   - `activations`: lista con las funciones de activación a aplicar en cada capa
3. En cada iteración:
   - Se aplica `linear_activation_forward`, que ejecuta el paso lineal y la activación.
   - Se guarda un **cache** con toda la información necesaria para calcular los gradientes en el *backpropagation*.
4. Tras procesar todas las capas, el valor final de `A` es:
   $$
   A^{[L]} = AL,
   $$
   la salida de la red neuronal.



#### Valores que devuelve

- **`AL`**  
  Activación de la última capa (probabilidad predicha para cada ejemplo).

- **`caches`**  
  Lista que contiene un cache por capa.  
  Cada cache incluye:
  - $A^{[l-1]}$
  - $W^{[l]}$
  - $b^{[l]}$
  - $Z^{[l]}$

Estos caches serán necesarios en la fase de retropropagación para calcular los gradientes capa a capa.


En resumen, `model_forward` ejecuta toda la secuencia de operaciones del *forward* de manera eficiente y completamente vectorizada, preparando todos los valores necesarios para el *backward*.


In [ ]:
def model_forward(X, params, activations):
    """
    X: (n_0, m)
    params: lista de capas [{"W":..., "b":...}, ...]
    activations: lista de funciones [relu, ..., sigmoid]
    
    devuelve:
      AL: activación de la última capa
      caches: lista de caches, uno por capa
    """
    A = X
    caches = []

    for layer, activation_fn in zip(params, activations):
        W = layer["W"]
        b = layer["b"]
        A, cache = linear_activation_forward(A, W, b, activation_fn)
        caches.append(cache)

    AL = A
    return AL, caches


#### Propagación hacia atras (*BackPropagation*)


La retropropagación implementa de forma vectorizada las ecuaciones calculan los gradientes de cada capa a partir de los valores obtenidos en la propagación hacia delante.  
Para cada capa $l$, el proceso sería:

1. Gradiente respecto a la preactivación $Z^{[l]}$: $\delta^{[l]}=\dfrac{\partial J}{\partial Z^{[l]}}$ 

$$
dZ^{[l]} = dA^{[l]} \odot g'^{[l]}(Z^{[l]})
$$


A partir del cálculo de  $dZ^{[l]}$ podemos calcular los gradientes respecto a los pesos y sesgos en esa capa

2. Gradiente respecto a los parámetros $W^{[l]}$ y $b^{[l]}$

$$
dW^{[l]} = \frac{1}{M}\, dZ^{[l]} A^{[l-1]\,T}
$$

$$
db^{[l]} = \frac{1}{M}\sum_{m=1}^{M} dZ^{[l]}_{:,m}
$$

Estos provienen directamente de las derivadas parciales de la ecuación:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$


3. Gradiente respecto a la activación de la capa anterior

$$
dA^{[l-1]} = W^{[l]\,T} dZ^{[l]}
$$

que es el gradiente que se propagará hacia la siguiente capa inferior. En la notación de la presentación $dA^{[l-1]}= W^{[l]\,T} \delta^{[l]}$



#### Implementación en código

Para traducir estas ecuaciones a Python, se utilizan cuatro funciones principales: una para la derivada de la función de activación ($g'^{[l]}(Z^{[l]})$), otra que calcula $dZ^{[l]}$, otra para las derivadas lineales a partir de $dZ^{[l]}$ y una cuarta que combina las anteriores.


1. Derivadas de activación


* Sigmoide  
```python
def sigmoid_prime(Z):
```
$$
g'(Z) = \sigma(Z)\,(1 - \sigma(Z))
$$


* ReLU 
```python
def relu_prime(Z):
```
$$
g'(Z) =
\begin{cases}
1 & \text{si } Z > 0 \\
0 & \text{si } Z \le 0
\end{cases}
$$

(En implementación: matriz de unos donde $Z>0$ y ceros donde $Z\le 0$.)

Estas funciones devuelven únicamente la **derivada de la activación**, sin mezclarla todavía con el gradiente entrante.

---

2. Combinación con el gradiente entrante
```python
def activation_backward(dA, Z, activation_prime):
```

El siguiente paso es combinar $dA^{[l]}$ (proveniente de la capa superior) con la derivada de la activación:

$$
dZ^{[l]} = dA^{[l]} \odot g'^{[l]}(Z^{[l]})
$$

Esta operación consiste simplemente en un producto elemento a elemento entre el gradiente que llega y la derivada de la activación evaluada en la capa.

---


3. Integración con la retropropagación lineal
```python
def linear_backward(dZ, cache):
```

Una vez obtenido $dZ^{[l]}$, se aplican las ecuaciones lineales:

$$
dW^{[l]} = \frac{1}{M} dZ^{[l]} A^{[l-1]\,T}
$$

$$
db^{[l]} = \frac{1}{M}\sum dZ^{[l]}
$$

$$
dA^{[l-1]} = W^{[l]\,T} dZ^{[l]}
$$

Estas ecuaciones utilizan exclusivamente valores almacenados en el `cache` de cada capa.

---

4. Por último una función que combina los dos componentes principales del *backpropagation* en una capa:

```python
def linear_activation_backward(dA, cache, activation_prime):
```

a) Derivada de la activación $dZ^{[l]}$

Este paso utiliza el gradiente entrante $dA^{[l]}$ y la derivada de la función de activación evaluada en $Z^{[l]}$.

b) A partir de $dZ^{[l]}$, se calculan $dW^{[l]}$, $db^{[l]}$ y $dA^{[l-1]}$

Estas expresiones calculan los gradientes respecto a los parámetros de la capa y el gradiente que debe propagarse hacia la capa anterior.



En conjunto, `linear_activation_backward` implementa de forma directa todas las ecuaciones teóricas del *backpropagation* para una capa de la red.


In [ ]:
def sigmoid_prime(Z):
    # ******** Tu codigo aqui ****************

    # ******** Tu codigo aqui ****************
    return Z_prime

def relu_prime(Z):
    # ******** Tu codigo aqui ****************

    # ******** Tu codigo aqui ****************
    return Z_prime

def activation_backward(dA, Z, activation_prime):
    # ******** Tu codigo aqui ****************
    
    # ******** Tu codigo aqui ****************
    return dZ

def linear_backward(dZ, cache):
    A_prev = cache["A_prev"]
    W = cache["W"]
    m = A_prev.shape[1]

    # ******** Tu codigo aqui ****************



    # ******** Tu codigo aqui ****************
    return dA_prev, dW, db

def linear_activation_backward(dA, cache, activation_prime):
    Z = cache["Z"]
    # dZ = dA * g'(Z)
    dZ = dA * activation_prime(Z)

    # Backward lineal
    dA_prev, dW, db = linear_backward(dZ, cache)
    return dA_prev, dW, db


### Retropropagación del modelo completo

La función `model_backward` implementa la **retropropagación** (*backpropagation*) a través de todas las capas de la red neuronal.  
A partir de la salida final del modelo $ AL $ y de las etiquetas reales $ Y $, calcula los gradientes necesarios para actualizar los parámetros de cada capa.

Recordemos que la retropropagación utiliza las ecuaciones derivadas del coste y de la estructura de cada capa:

1. **Cálculo del gradiente del coste respecto a la salida**  
   En el caso de entropía cruzada binaria:
   $$
   \frac{\partial J}{\partial A^{[L]}} 
   = -\left( \frac{Y}{A^{[L]}} - \frac{1-Y}{1-A^{[L]}} \right)
   $$

2. **Paso backward en cada capa**  
   Cada capa combina:
   - Derivada de la activación  
     $$
     dZ^{[l]} = dA^{[l]} \odot g'^{[l]}(Z^{[l]})
     $$
   - Derivadas lineales  
     $$
     dW^{[l]} = \frac{1}{m} dZ^{[l]} A^{[l-1]\,T}
     \qquad
     db^{[l]} = \frac{1}{m} \sum dZ^{[l]}
     \qquad
     dA^{[l-1]} = W^{[l]\,T} dZ^{[l]}
     $$



1. Obtiene el número de ejemplos y el número total de capas.
2. Inicializa una lista `grads` donde se guardarán los gradientes de cada capa.
3. Calcula el gradiente inicial:
   $$
   dA^{[L]} = \frac{\partial J}{\partial A^{[L]}}
   $$
   utilizando la fórmula de la entropía cruzada.
4. Recorre las capas **en orden inverso** (de la última a la primera):
   - Recupera el `cache` almacenado en el forward.
   - Llama a `linear_activation_backward` para calcular  
     $dW^{[l]}, db^{[l]}, dA^{[l-1]}$.
   - Almacena los gradientes en `grads[l]`.



#### Valores que devuelve

- **`grads`**  
  Lista de diccionarios con los gradientes de cada capa, donde cada elemento contiene:
  ```python
  {"dW": dW_l, "db": db_l}
  ```

Estos valores se utilizarán en la función update_params para actualizar pesos y sesgos mediante descenso de gradiente.

En resumen, `model_backward` implementa de forma totalmente vectorizada todas las ecuaciones del backpropagation, recorriendo la red desde la capa de salida hasta la capa de entrada, y devolviendo los gradientes necesarios para el aprendizaje del modelo.


In [ ]:
def model_backward(AL, Y, caches, activations_prime):
    """
    AL: salida de la red (n_L, m)
    Y:  etiquetas (n_L, m)
    caches: lista de caches del forward
    activations_prime: lista [relu_prime, ..., sigmoid_prime]
    
    devuelve:
      grads: lista de diccionarios [{"dW":..., "db":...}, ...]
    """
    m = Y.shape[1]
    L = len(caches)

    grads = [None] * L

    # dA de la última capa (cross_entropy)
    dA = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL))


    # Recorremos capas hacia atrás
    for l in reversed(range(L)):
        cache = caches[l]
        dA, dW, db = linear_activation_backward(
            dA,
            cache,
            activations_prime[l]
        )
        grads[l] = {"dW": dW, "db": db}

    return grads


### Actualización de los parámetros mediante descenso de gradiente

La función `update_params` implementa el paso final del ciclo de aprendizaje de la red neuronal:  
la **actualización de los pesos y sesgos** utilizando los gradientes calculados durante el *backpropagation*.

Recordemos que en descenso de gradiente los parámetros de cada capa se ajustan según:

$$
W^{[l]} \leftarrow W^{[l]} - \alpha \, dW^{[l]}
$$
$$
b^{[l]} \leftarrow b^{[l]} - \alpha \, db^{[l]}
$$

donde:

- $ W^{[l]} $, $ b^{[l]} $ son los parámetros actuales de la capa $l$  
- $ dW^{[l]} $, $ db^{[l]} $ son los gradientes obtenidos en el *backward*  
- $ \alpha $ es la tasa de aprendizaje (`learning_rate`)



La función realiza exactamente estas actualizaciones:

1. Recorre todas las capas de la red mediante un bucle.
2. Para cada capa:
   - Resta al peso $W^{[l]}$ el gradiente escalado por la tasa de aprendizaje.
   - Resta al sesgo $b^{[l]}$ el gradiente correspondiente.
3. Devuelve la estructura `params` ya actualizada.

Esto completa una iteración del ciclo:

$$
\text{forward} \;\rightarrow\; \text{coste} \;\rightarrow\; 
\text{backward} \;\rightarrow\; \text{update}
$$

permitiendo que la red neuronal mejore su rendimiento con cada época de entrenamiento.


In [ ]:
def update_params(params, grads, learning_rate):
    for l in range(len(params)):
        params[l]["W"] -= learning_rate * grads[l]["dW"]
        params[l]["b"] -= learning_rate * grads[l]["db"]
    return params


### Cálculo de la función de coste (entropía cruzada binaria)

La función `compute_cost` calcula el **coste de entropía cruzada binaria**, que es la medida estándar utilizada en problemas de **clasificación binaria** como el de esta práctica (*gato vs no gato*).

Dado:

- $ AL $: salida de la red neuronal (probabilidades), de dimensión $(1, m)$  
- $ Y $: etiquetas reales (0 = no gato, 1 = gato), también de dimensión $(1, m)$

el coste se define como:

$$
J = -\frac{1}{m} \sum_{i=1}^{m}
\left[
Y^{(i)} \log(A_L^{(i)}) +
(1 - Y^{(i)}) \log(1 - A_L^{(i)})
\right]
$$

Esta función mide qué tan bien las predicciones del modelo se ajustan a las etiquetas reales:  
- Si $ AL $ coincide con $ Y $, el coste es bajo.  
- Si se aleja de las etiquetas reales, el coste crece.



### ¿Qué hace la función?

1. Obtiene el número de ejemplos $m$.
2. Calcula el coste según la fórmula anterior usando operaciones vectorizadas.
3. Aplica `np.squeeze` para convertir el coste en un escalar (elimina dimensiones de tamaño 1).
4. Comprueba que el resultado final tiene la forma adecuada.

Este valor de coste se utiliza durante el entrenamiento para monitorizar el aprendizaje de la red y asegurarse de que está convergiendo correctamente.


In [ ]:
def compute_cost(AL, Y):
    """
    Arguments:
    AL -- probability vector corresponding to your label predictions, shape (1, number of examples)
    Y -- true "label" vector (for example: containing 0 if non-cat, 1 if cat), shape (1, number of examples)

    Returns:
    cost -- cross-entropy cost
    """
    
    m = Y.shape[1]

    # Calculo de la entropia cruzada
    cost = (1./m) * (-np.dot(Y,np.log(AL).T) - np.dot(1-Y, np.log(1-AL).T))
    
    cost = np.squeeze(cost)      # Se deshace de las dimensiones innecesarias (e.g. this turns [[17]] into 17).
    assert(cost.shape == ())
    
    return cost

# Copia a continuación todas las funciones del cuaderno en la celda de abajo. 

## Al ejecutarla se creará un fichero `nn_auxiliar.py` utilizado por el cuaderno principal

In [ ]:
%%writefile nn_auxiliar.py

import numpy as np
import matplotlib.pyplot as plt
import h5py



Overwriting nn_auxiliar.py
